Imports


In [ ]:
import numpy as np
import functools, operator
import string


In [ ]:
N_INPUTS: int = 3
N_LAYERS: int = 3
N_NODESPERLAYER: int = 8
N_OUTPUTS: int = 1

rng = np.random.default_rng(42)

inWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_NODESPERLAYER, N_INPUTS])
inBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)), N_NODESPERLAYER)
layerWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_LAYERS - 1, N_NODESPERLAYER, N_NODESPERLAYER])
layerBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_LAYERS - 1, N_NODESPERLAYER])
outWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_OUTPUTS, N_NODESPERLAYER])
outBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)))

def relu(x):
    return np.maximum(0, x)

# Forwarpass
def RunNetwork(_input, _inWeights, _inBias, _layerWeights, _layerBias, _outWeights, _outBias):
    act = relu(_inWeights @ _input + _inBias)  # input linear + relu
    for W, b in zip(_layerWeights, _layerBias):
        act = relu(W @ act + b)  # loop over layers: linear + relu
    act = _outWeights @ act + _outBias  # linear output layer
    return act


RunNetwork(rng.random(3), inWeights, inBias, layerWeights, layerBias, outWeights, outBias)

In [ ]:
class Tensor:
    def __init__(self, data: np.ndarray | list):
        if type(data) == np.ndarray:
            self.data = data
        else:
            self.data = np.array(data, dtype=np.double)
        self.shape = np.shape(self.data)
        self.grad = np.zeros(self.shape)
        self._backward = lambda: None


    def __repr__(self):
        return f"Tensor(data=\n{self.data})"

    def reshape(self, newShape):
        self.data.reshape(newShape)
        self.shape = np.shape(self.data)

    # --- Rechenoperationen ---
    # Komponentenweise Addition
    def __add__(self, other):
        out = Tensor(self.data + other.data)

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    # Matrix-Matrix Multiplikation
    def __matmul__(self, other):
        out = Tensor(self.data @ other.data)

        def _backward():
            
            self.grad += 0
            other.grad += 0

        out._backward = _backward
        return out

    # Komponentenweise Multiplikation
    def __mul__(self, other):
        out = Tensor(self.data * other.data)

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out


In [ ]:
def insert(tensor: Tensor, insertion: tuple[int, ...]):
    return Tensor(np.expand_dims(tensor.data, insertion))

def stretch(tensor: Tensor, insertion: tuple[int, ...], length: tuple[int, ...]) -> Tensor:
    array = np.expand_dims(tensor.data, insertion)

    shape = list(tensor.shape)
    for i, val in zip(insertion, length): 
        shape.insert(i, val)
    out = Tensor(np.broadcast_to(array, shape))
    return out

def contraction(tensors: list[Tensor], schema: str):
    return Tensor(np.einsum(schema, *[t.data for t in tensors]))

def partial_jacobian(tensors: list[Tensor], schema: str, targetPos: int):
    # Determine unused letters in schema
    lhs, rhs = schema.replace(" ", "").split("->")
    inds = lhs.split(",")
    l_used, r_used = set(lhs.replace(",", "")), set(rhs)
    unused = list(c for c in string.ascii_letters if c not in l_used | r_used)
    unused.sort()
    
    # Remove target tensor from indices 
    _tensors = list(tensors)
    target = _tensors.pop(targetPos)
    targetInd = inds.pop(targetPos)

    # Construct in- and output indices
    outInd = rhs
    trailing_dims = {}
    for pos, (i, p) in enumerate(zip(targetInd, unused)):
        # Construct output index
        outInd += p
        trailing_dims[p] = target.shape[pos]

        # Modify input indices
        # Case: Index* in f-Block
        if i in rhs:
            axis_length = target.shape[pos]
            _tensors.append(Tensor(np.identity(axis_length)))
            inds.append(i + p)
        
        # Case: Index* not in f-Block
        else:
            inds = [ind.replace(i, p) for ind in inds]

    # Determine indices and their positions not occuring in input
    inds_set = set("".join(inds))
    missing = set(outInd) - inds_set

    insertions = tuple(pos for pos, i in enumerate(outInd) if i in missing)
    occuring_out = "".join([i for i in outInd if i in inds_set])

    # Calculate contraction with occuring indices
    if inds: # to prevent empty einsum
        occuring_result = np.einsum(f"{",".join(inds)} -> {occuring_out}", *[t.data for t in _tensors])
    else:
        occuring_result = np.array(1.0)

    # Broadcast missing indices to right length
    final_shape = (trailing_dims[p] if p in trailing_dims else np.shape(occuring_result)[pos] for pos, p in enumerate(outInd))

    inserted_result = np.expand_dims(occuring_result, insertions)
    final_result = np.broadcast_to(inserted_result, final_shape)

    return Tensor(final_result)

def partial_gradient(tensors: list[Tensor], schema: str, targetPos: int, outGrad: Tensor):
    lhs, rhs = schema.replace(" ", "").split("->")
    inds = lhs.split(",")

    # Remove target tensor from indices 
    _tensors = list(tensors)
    target = _tensors.pop(targetPos)
    targetInd = inds.pop(targetPos)

    # Construct in- and output indices
    outInd = targetInd
    inds.append(rhs)
    _tensors.append(outGrad)
    
    # Determine indices and their positions not occuring in input
    inds_set = set("".join(inds))
    missing = set(outInd) - inds_set

    insertions = tuple(pos for pos, i in enumerate(outInd) if i in missing)
    occuring_out = "".join([i for i in outInd if i in inds_set])

    # Calculate contraction with occuring indices
    if inds: # to prevent empty einsum
        occuring_result = np.einsum(f"{",".join(inds)} -> {occuring_out}", *[t.data for t in _tensors])
    else:
        occuring_result = np.array(1.0)

    # Broadcast missing indices to right length
    final_shape = target.shape

    inserted_result = np.expand_dims(occuring_result, insertions)
    final_result = np.broadcast_to(inserted_result, final_shape)

    return Tensor(final_result)

In [ ]:
rng = np.random.default_rng(0)

A = Tensor(np.arange(9.0).reshape([3,3]))
B = Tensor(rng.random([3,3], dtype=np.float32))
x = Tensor([1,10,100])
y = Tensor([1,2,3])

arguments = [A,B]
schema = "ij,jk->ik"
outGradient = partial_jacobian([contraction(arguments, schema)], "ij->", 0)

print("Map: ", contraction(arguments, schema))

# print("\nPartial jacobian i.r.t. argument 0:", partial_jacobian(arguments, schema, 0))
# print("\nPartial jacobian i.r.t. argument 1:", partial_jacobian(arguments, schema, 1))

print("\nPartial gradient i.r.t. argument 0:", partial_gradient(arguments, schema, 0, outGradient))
print("\nPartial gradient i.r.t. argument 1:", partial_gradient(arguments, schema, 1, outGradient))


In [ ]:
# x = Tensor([1,1])
# y = Tensor([1, 10, 100])

x = Tensor(np.arange(9.0).reshape([3,3]))
y = Tensor(np.identity(3))
y.data[1,1] = -1.0
tensMult([x, y], "ij, jk -> ik")


# x = np.arange(9.0).reshape([3,3])
# np.einsum("ii->", x) 


In [ ]:
x = Tensor(np.arange(3.0))  # [0, 1, 2]

y = Tensor(np.array([1, 10, 100]))  # [1, 10, 100]

x = stretch(x, (0,), (3,))  # [[0, 0, 0], [1, 1, 1], [2, 2, 2]]

y = stretch(y, (0,), (3,))

z = contract(x * y, (0,))

z

In [ ]:
x = np.arange(9).reshape((3,3))

y = np